In [3]:
import pandas as pd
path = "data/dados.xlsx"
df = pd.read_excel(path)

In [4]:
desc_list = list(df["descricao"].unique())
desc_list

['Transação referente à contratação de pacote premium com acesso ampliado a recursos avançados, relatórios detalhados, integração com sistemas externos e suporte técnico dedicado durante todo o ciclo contratado.',
 'Cobrança vinculada à utilização de serviços recorrentes prestados ao cliente, contemplando manutenção da infraestrutura, monitoramento de desempenho e atendimento especializado conforme contrato vigente estabelecido previamente.',
 'Pagamento referente à assinatura mensal de serviços digitais contratados pela empresa, incluindo acesso à plataforma, suporte técnico contínuo e atualizações periódicas com novos recursos e melhorias no sistema.',
 'Pagamento de serviço',
 'Pagamento efetuado pela aquisição de serviços adicionais não inclusos no plano básico, envolvendo customizações específicas, análises detalhadas e suporte técnico especializado sob demanda conforme necessidade do cliente.',
 'Valor correspondente à renovação de licença anual de software corporativo, incluindo

In [5]:
from app.core.llm import llm_client

llm_client.initialize()

llm = llm_client.get_llm()


In [ ]:
# prompt para saber se essa transação aparenta ter natureza recorrente ou não

def build_recurrence_classification_prompt(description: str, values: str) -> str:

    values_str = f"\n- ".join(values)

    return f"""
Você é um classificador de transações financeiras.

Sua tarefa é classificar se uma transação aparenta ter natureza recorrente ou não.

RECORRÊNCIA = indica se há periodicidade

VALORES POSSÍVEIS:
- recorrente
- isolado

REGRAS:
- escolha apenas UM valor da lista
- se não houver indicação clara, responda "desconhecido"
- não invente novos valores
- responda apenas o valor (sem explicação)

DESCRIÇÃO:
{description}
"""

In [8]:
desc = desc_list[0]
desc

'Transação referente à contratação de pacote premium com acesso ampliado a recursos avançados, relatórios detalhados, integração com sistemas externos e suporte técnico dedicado durante todo o ciclo contratado.'

In [16]:
for desc in desc_list:
    prompt = build_recurrence_classification_prompt(desc)
    response = llm.complete(prompt)
    print(f"{desc[:40]} : {response.text.strip().lower()}")

Transação referente à contratação de pac : recorrente
Cobrança vinculada à utilização de servi : recorrente
Pagamento referente à assinatura mensal  : recorrente
Pagamento de serviço : desconhecido
Pagamento efetuado pela aquisição de ser : recorrente
Valor correspondente à renovação de lice : recorrente
Plano premium : recorrente
Licença anual : recorrente
Assinatura mensal : recorrente
Compra única : isolado


## Frequencia

In [22]:
def build_prompt_freq(description:str) -> str:
    return f"""
Você é um analista financeiro.

Classifique a frequência da transação.

Valores permitidos

VALORES POSSÍVEIS:
- mensal
- anual
- sob_demanda
- nao_aplicavel

REGRAS:
- escolha apenas UM valor da lista
- se não houver indicação clara, responda "Não aplicável"
- não invente novos valores
- responda apenas o valor (sem explicação)

DESCRIÇÃO:
{description}
"""

In [23]:
for desc in desc_list:
    prompt = build_prompt_freq(desc)
    response = llm.complete(prompt)
    print(f"{desc[:40]} : {response.text.strip().lower()}")

Transação referente à contratação de pac : sob_demanda
Cobrança vinculada à utilização de servi : "mensal"
Pagamento referente à assinatura mensal  : mensal
Pagamento de serviço : não aplicável
Pagamento efetuado pela aquisição de ser : sob_demanda
Valor correspondente à renovação de lice : anual
Plano premium : não aplicável (nao_aplicavel)
Licença anual : anual
Assinatura mensal : mensal
Compra única : não aplicável
